# Task 4: Đẩy Đồ thị vào Neo4j

## 1. Phương pháp luận và Lý do thiết kế

Ở bước này, mục tiêu là đưa các sự kiện hình thành cấu trúc đồ thị (Nodes và Edges) từ Kafka vào cơ sở dữ liệu đồ thị **Neo4j**.

**Sử dụng Neo4j Kafka Connector (Sink):**
Thay vì viết một ứng dụng Apache Spark Streaming phức tạp làm lớp trung gian, chúng tôi lựa chọn sử dụng Plugin chính thức `neo4j-kafka-connect` của Neo4j. Lý do là:
- Dữ liệu Node và Edge đã được Parser định dạng chuẩn JSON, có thể đổ trực tiếp vào DB mà không cần biến đổi thêm (Zero-ETL).
- Tiết kiệm tài nguyên xử lý cho cụm Spark, để Spark chuyên tâm vào việc xử lý Metadata (Task 5).

**Tính Lũy đẳng (Idempotency):**
Đặc tả yêu cầu đòi hỏi hệ thống không được sinh ra dữ liệu trùng lặp nếu một file Python được parse lại. Để giải quyết triệt để vấn đề này:
1. Parser sinh ra một ID cố định (`node_id` băm từ filepath+line, `edge_id` băm từ node nguồn+đích).
2. Kafka Sink Connector được cấu hình sử dụng mệnh đề **`MERGE`** thay vì `CREATE` trong Cypher:
   - Đối với Node: `MERGE (n:ASTNode {node_id: event.node_id}) SET n += event`
   - Đối với Edge: `MATCH (src), (tgt) MERGE (src)-[r:EDGE]->(tgt)`
Lệnh `MERGE` đảm bảo rằng nếu node/edge đã tồn tại, nó chỉ ghi đè thuộc tính chứ không sinh ra node mới, đáp ứng 100% yêu cầu Idempotency.


In [ ]:
# Ô Notebook thực thi: Gọi REST API của Kafka Connect để kiểm tra trạng thái Connector
# YÊU CẦU: HÃY CHẠY Ô CODE NÀY ĐỂ MINH CHỨNG SINK CONNECTOR ĐANG HOẠT ĐỘNG!
import urllib.request
import json

urls = [
    "http://127.0.0.1:8083/connectors/neo4j-sink-node/status",
    "http://127.0.0.1:8083/connectors/neo4j-sink-edge/status"
]

for url in urls:
    try:
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode('utf-8'))
            name = data.get("name")
            connector_state = data.get("connector", {}).get("state")
            print(f"🔌 Connector '{name}': State = {connector_state}")
    except Exception as e:
        print(f"Không thể kết nối đến {url}. Lỗi: {e}")


## 2. Giao diện Đồ thị Neo4j (Neo4j Browser)

Dưới đây là hình ảnh minh chứng dữ liệu đã được đẩy thành công vào Neo4j và tạo thành mạng lưới đồ thị CPG.

**Hình 1: Dữ liệu Node (Các thành phần mã nguồn) trong Neo4j**
![Neo4j Nodes](images/task4_1_1.png)

**Hình 2: Dữ liệu Edge (Các liên kết Gọi hàm, Luồng điều khiển) trong Neo4j**
![Neo4j Edges](images/task4_1_2.png)


## 3. Reflection 
**Những gì hiệu quả:**
- Giải pháp dùng `Kafka Connect` tỏ ra vô cùng hiệu quả. Nó hoạt động âm thầm và bền bỉ trong nền, tự động hút dữ liệu tốc độ rất cao mà team không phải viết bất kỳ dòng mã xử lý dữ liệu nào (chỉ viết cấu hình JSON).

**Những gì gặp khó khăn & Cách giải quyết:**
- Việc viết câu lệnh Cypher để nhúng vào JSON cấu hình của Kafka Sink rất dễ sai cú pháp, và không có công cụ debug trực tiếp.
- Lệnh `MERGE` cần thời gian để tìm kiếm Node trong DB. Khi số lượng Node lên tới hàng triệu, tốc độ ghi bị giảm thê thảm. 
- *Cách giải quyết:* Chúng tôi đã phải tạo Index trong Neo4j cho thuộc tính `node_id` (`CREATE INDEX FOR (n:ASTNode) ON (n.node_id);`) để tăng tốc độ lệnh MERGE lên hàng trăm lần.
